# Robust costs – Adaptive IRLS

Using M-estimators with **adaptive scale estimation** via the built-in `irls_weight_fn` API.

The [fixed-parameter robust costs notebook](robust_costs.ipynb) implements IRLS
manually via `jax.lax.stop_gradient`.  That approach requires choosing a scale
threshold in the same units as the residuals — a parameter that depends on the
problem.

jaxls provides a cleaner, **built-in** path: pass an `irls_weight_fn` to
{func}`@jaxls.Cost.factory <jaxls.Cost.factory>`.  The solver calls the weight
function at every iteration with the **full group** of residuals, applies
`sqrt(w)` to both the residual vector and the Jacobian rows, and solves the
resulting weighted normal equations.  No `stop_gradient` wiring is needed.

The **adaptive** variants go one step further: they estimate the noise scale
**σ** from the current residuals at each iteration using the Median Absolute
Deviation (MAD),

$$\hat{\sigma} = \frac{\operatorname{median}(|r|)}{0.6745}$$

and compute weights from the scale-normalised residuals $u_i = r_i / \hat{\sigma}$.
The constant 0.6745 makes MAD a consistent estimator of the standard deviation
under Gaussian noise ($E[\text{MAD}] = 0.6745\,\sigma$).  Because the scale is
re-estimated each iteration, the estimator adapts to whatever noise level is
present — no manual tuning required.

### Available weight functions

| | Fixed scale | Adaptive scale |
|---|---|---|
| **Huber** | `irls_huber(delta)` | `irls_huber_adaptive(k)` |
| **Cauchy** | `irls_cauchy(c)` | `irls_cauchy_adaptive(k)` |
| **Tukey bisquare** | `irls_tukey(c)` | `irls_tukey_adaptive(k)` |
| **Geman-McClure** | `irls_geman_mcclure(c)` | `irls_geman_mcclure_adaptive(k)` |
| **Welsh** | `irls_welsh(c)` | `irls_welsh_adaptive(k)` |
| **L_p norm** | `irls_lp(p, eps)` | `irls_lp_adaptive(p, eps)` |
| **L1** | `irls_l1(eps)` | — |

Features used:
- {class}`~jaxls.Var` subclass for circle parameters
- {func}`@jaxls.Cost.factory <jaxls.Cost.factory>` with `irls_weight_fn`
- Adaptive M-estimator factories from `jaxls.utils`

In [1]:
import sys
from loguru import logger

logger.remove()
logger.add(sys.stdout, format="<level>{level: <8}</level> | {message}");

In [2]:
import jax
import jax.numpy as jnp
import jaxls
import numpy as np

## The outlier problem

Consider fitting a circle to 2D points.  With clean data, least squares works
well.  But real-world data often contains outliers — points that don't follow
the expected model due to sensor errors, misassociations, or other anomalies.

Standard least squares minimizes the sum of squared residuals:

$$\min_\theta \sum_i r_i(\theta)^2$$

The squaring amplifies large residuals, giving outliers disproportionate
influence.

In [3]:
# Generate synthetic circle data with outliers.
np.random.seed(42)

# Ground truth circle.
true_cx, true_cy, true_r = 1.0, 1.0, 2.0

# Inlier points (on the circle with small noise).
n_inliers = 40
theta_inliers = np.random.uniform(0, 2 * np.pi, n_inliers)
noise_inliers = np.random.normal(0, 0.1, n_inliers)
inlier_x = true_cx + (true_r + noise_inliers) * np.cos(theta_inliers)
inlier_y = true_cy + (true_r + noise_inliers) * np.sin(theta_inliers)

# Outlier points (scattered far from the circle).
n_outliers = 10
outlier_x = np.random.uniform(-4, 7, n_outliers)
outlier_y = np.random.uniform(-4, 7, n_outliers)

# Combine all points.
all_x = np.concatenate([inlier_x, outlier_x])
all_y = np.concatenate([inlier_y, outlier_y])
points = jnp.stack([all_x, all_y], axis=-1)
n_points = len(points)

# Track which points are outliers for visualization.
is_outlier = np.array([False] * n_inliers + [True] * n_outliers)

print(
    f"Generated {n_inliers} inliers and {n_outliers} outliers "
    f"({n_outliers / n_points * 100:.0f}% outliers)"
)
print(f"True circle: center=({true_cx}, {true_cy}), radius={true_r}")

Generated 40 inliers and 10 outliers (20% outliers)
True circle: center=(1.0, 1.0), radius=2.0


## Standard least squares baseline

First, solve the unweighted problem so we can see how outliers pull the
estimate away from the true circle:

In [4]:
class CircleVar(
    jaxls.Var[jax.Array], default_factory=lambda: jnp.array([0.0, 0.0, 1.0])
):
    """Circle parameters: [cx, cy, r]."""


@jaxls.Cost.factory
def circle_residual(
    vals: jaxls.VarValues,
    circle: CircleVar,
    point: jax.Array,
) -> jax.Array:
    """2D residual: error vector from closest circle point to observed point."""
    params = vals[circle]
    cx, cy, r = params[0], params[1], params[2]
    diff = point - jnp.array([cx, cy])
    dist = jnp.sqrt(jnp.sum(diff**2) + 1e-8)
    direction = diff / dist
    return (dist - r) * direction

In [5]:
circle_var = CircleVar(id=0)

# Initial guess: centroid of points, average distance as radius.
centroid = jnp.mean(points, axis=0)
avg_dist = jnp.mean(jnp.sqrt(jnp.sum((points - centroid) ** 2, axis=-1)))
initial_params = jnp.array([centroid[0], centroid[1], avg_dist])

costs_standard = [
    circle_residual(CircleVar(id=jnp.zeros(n_points, dtype=jnp.int32)), points)
]
initial_vals = jaxls.VarValues.make([circle_var.with_value(initial_params)])
problem_standard = jaxls.LeastSquaresProblem(costs_standard, [circle_var]).analyze()
solution_standard = problem_standard.solve(initial_vals)

params_standard = solution_standard[circle_var]
print(
    f"Standard LS result: center=({params_standard[0]:.3f}, {params_standard[1]:.3f}), "
    f"radius={params_standard[2]:.3f}"
)
print(f"True parameters:    center=({true_cx:.3f}, {true_cy:.3f}), radius={true_r:.3f}")

INFO     | Building optimization problem with 50 terms and 1 variables: 50 costs, 0 eq_zero, 0 leq_zero, 0 geq_zero
INFO     | Vectorizing group with 50 costs, 1 variables each: circle_residual
INFO     | Variable elimination: eliminating CircleVar (3 of 3 tangent dims); reduced system is 0-dimensional
INFO     | Variable elimination: every type was eliminated, so the Hessian is fully block-diagonal; the step is an exact blockwise inverse and the linear_solver choice is ignored.
INFO     | The Hessian is fully block-diagonal (every variable type was eliminated), so linear_solver='conjugate_gradient' is ignored; the step is solved by exact blockwise inversion.
INFO     |  step #0: cost=62.2762 lambd=0.0005 inexact_tol=1.0e-02
INFO     |      - circle_residual(50): 62.27618 (avg 0.62276)
INFO     |      accepted=True ATb_norm=8.01e+00 cost_prev=62.2762 cost_new=60.3239
INFO     |  step #1: cost=60.3239 lambd=0.0003 inexact_tol=1.0e-02
INFO     |      - circle_residual(50): 60.32395 (avg 

## Adaptive M-estimators

M-estimators replace the squared loss $r^2$ with a robust function $\rho(r)$
that grows more slowly for large residuals.  In the IRLS framework this is
equivalent to solving a sequence of *weighted* least-squares problems
$\sum_i w_i\, r_i^2$, where the weights $w_i$ depend on the current residuals.

The **adaptive** variants normalise each residual by $\hat{\sigma}$ before
computing the weight.  The six built-in adaptive factories and their weight
formulas (with normalised residual $u = r / \hat{\sigma}$) are:

**Huber** — quadratic for small residuals, linear for large:

$$w_i = \begin{cases}1 & |u_i| \le k \\ k\,/\,|u_i| & |u_i| > k\end{cases}
\qquad (\text{default } k = 1.345)$$

**Cauchy / Lorentzian** — smooth down-weighting, never fully rejects:

$$w_i = \frac{1}{1 + (u_i / k)^2}
\qquad (\text{default } k = 2.385)$$

**Tukey bisquare** — hard rejection beyond $k\,\hat{\sigma}$:

$$w_i = \begin{cases}\bigl(1 - (u_i / k)^2\bigr)^2 & |u_i| \le k \\ 0 & |u_i| > k\end{cases}
\qquad (\text{default } k = 4.685)$$

**Geman-McClure** — very aggressive down-weighting (squared Cauchy decay):

$$w_i = \frac{1}{\bigl(1 + (u_i / k)^2\bigr)^2}
\qquad (\text{default } k = 3.0)$$

**Welsh** (Dennis–Welsch) — smooth exponential decay, redescending but never exactly zero:

$$w_i = \exp\!\bigl(-(u_i / k)^2\bigr)
\qquad (\text{default } k = 2.985)$$

**L_p norm** — generalised $\ell_p$ minimiser for $0 < p \le 2$:

$$w_i = |u_i|^{\,p - 2}
\qquad (\text{default } p = 1.2)$$

When $p = 2$ the weights are all 1 (standard L2).  As $p$ decreases toward 0,
large residuals are down-weighted more aggressively.  $p = 1$ recovers L1
(median-like) behaviour.

The default $k$ values for Huber, Cauchy, Tukey, and Welsh are chosen so that
each estimator achieves **95 % asymptotic efficiency** relative to ordinary
least squares under Gaussian noise — a standard rule of thumb in robust
statistics.

In [6]:
import plotly.graph_objects as go
from IPython.display import HTML


def make_circle_trace(
    cx: float, cy: float, r: float, name: str, color: str, dash: str = "solid"
) -> go.Scatter:
    """Create a circle trace for plotting."""
    theta = np.linspace(0, 2 * np.pi, 100)
    x = cx + r * np.cos(theta)
    y = cy + r * np.sin(theta)
    return go.Scatter(
        x=x,
        y=y,
        mode="lines",
        name=name,
        line=dict(color=color, width=2, dash=dash),
    )

### Visualising the weight curves

The plot below shows how each adaptive weight function responds to normalised
residuals $u = r / \hat{\sigma}$.

- **Huber** transitions sharply from weight 1 to $k/|u|$ at $|u| = k$.
- **Cauchy** decays smoothly — no residual is fully ignored.
- **Geman-McClure** decays as the *square* of Cauchy, giving much stronger
  suppression of large residuals.
- **Welsh** decays exponentially — extremely fast fall-off, but the weight
  never reaches exactly zero.
- **Tukey** drops to *exactly* zero beyond $|u| = k$, completely excluding
  gross outliers.
- **L_p** ($p = 1.2$) provides gentle, power-law down-weighting without a
  sharp threshold.

In [7]:
u_vals = jnp.linspace(-8, 8, 400)
u_2d = u_vals.reshape(1, -1)  # (1, 400) — group format expected by irls_weight_fn

w_huber = jaxls.utils.irls_huber_adaptive(k=1.345)(u_2d)[0]
w_cauchy = jaxls.utils.irls_cauchy_adaptive(k=2.385)(u_2d)[0]
w_tukey = jaxls.utils.irls_tukey_adaptive(k=4.685)(u_2d)[0]
w_gm = jaxls.utils.irls_geman_mcclure_adaptive(k=3.0)(u_2d)[0]
w_welsh = jaxls.utils.irls_welsh_adaptive(k=2.985)(u_2d)[0]
w_lp = jaxls.utils.irls_lp_adaptive(p=1.2)(u_2d)[0]

fig_weights = go.Figure()
fig_weights.add_trace(
    go.Scatter(
        x=u_vals, y=jnp.ones_like(u_vals),
        mode="lines", name="Standard LS (w=1)",
        line=dict(color="gray", dash="dash"),
    )
)
for label, w, color in [
    ("Huber (k=1.345)", w_huber, "#2196F3"),
    ("Cauchy (k=2.385)", w_cauchy, "#4CAF50"),
    ("Geman-McClure (k=3.0)", w_gm, "#E91E63"),
    ("Welsh (k=2.985)", w_welsh, "#9C27B0"),
    ("Tukey (k=4.685)", w_tukey, "#FF9800"),
    ("Lp (p=1.2)", w_lp, "#795548"),
]:
    fig_weights.add_trace(
        go.Scatter(x=u_vals, y=w, mode="lines", name=label, line=dict(color=color))
    )

fig_weights.update_layout(
    title="Adaptive Weight Functions (normalised residual u = r / σ̂)",
    xaxis_title="Normalised residual u",
    yaxis_title="Weight w(u)",
    height=400,
    margin=dict(t=40, b=40, l=60, r=40),
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
)
HTML(fig_weights.to_html(full_html=False, include_plotlyjs="cdn"))

## Using `irls_weight_fn` in jaxls

The built-in `irls_weight_fn` API replaces the manual `stop_gradient` pattern
from the [fixed-parameter notebook](robust_costs.ipynb).  There are two
differences:

1. **The residual function stays clean** — it returns the plain geometric
   residual, with no weighting logic.
2. **The solver handles everything** — at each iteration it calls
   `irls_weight_fn(residuals)` with the full `(count, residual_flat_dim)`
   array, then multiplies both the residual vector and Jacobian rows by
   `sqrt(w)` so the normal equations become $J^T W J\,\Delta x = -J^T W r$.

Usage is a single keyword argument:

```python
@jaxls.Cost.factory(irls_weight_fn=jaxls.utils.irls_cauchy_adaptive())
def my_cost(vals, var, data):
    return plain_residual(vals, var, data)
```

In [8]:
def make_adaptive_robust_circle_cost(irls_weight_fn):
    """Return a Cost factory that applies the given adaptive IRLS weight function."""

    @jaxls.Cost.factory(irls_weight_fn=irls_weight_fn)
    def robust_circle_residual(
        vals: jaxls.VarValues,
        circle: CircleVar,
        point: jax.Array,
    ) -> jax.Array:
        """Plain geometric residual -- weighting is handled by irls_weight_fn."""
        params = vals[circle]
        cx, cy, r = params[0], params[1], params[2]
        diff = point - jnp.array([cx, cy])
        dist = jnp.sqrt(jnp.sum(diff**2) + 1e-8)
        direction = diff / dist
        return (dist - r) * direction

    return robust_circle_residual


def compute_scalar_weights(
    params: jax.Array, points: jax.Array, weight_fn
) -> jax.Array:
    """Compute a single scalar weight per point for visualisation.

    weight_fn follows the irls_weight_fn group signature: (count, dim) -> (count, dim).
    We reduce the per-element weights to per-point scalars by taking the mean.
    """
    cx, cy, r = params[0], params[1], params[2]
    dists = jnp.sqrt(
        (points[:, 0] - cx) ** 2 + (points[:, 1] - cy) ** 2 + 1e-8
    )
    # Build a 2D residual matching circle_residual's output shape.
    diff = points - jnp.array([cx, cy])
    direction = diff / dists[:, None]
    residual_2d = (dists[:, None] - r) * direction  # (n, 2)
    weights_2d = weight_fn(residual_2d)  # (n, 2)
    return jnp.mean(weights_2d, axis=-1)  # (n,)

## Robust fitting with adaptive Cauchy weights

A single `solve()` call — no manual tuning of the scale parameter:

In [9]:
cauchy_adaptive_cost = make_adaptive_robust_circle_cost(
    jaxls.utils.irls_cauchy_adaptive()
)

costs_cauchy = [
    cauchy_adaptive_cost(
        CircleVar(id=jnp.zeros(n_points, dtype=jnp.int32)), points
    )
]
problem_cauchy = jaxls.LeastSquaresProblem(costs_cauchy, [circle_var]).analyze()
solution_cauchy = problem_cauchy.solve(initial_vals)

params_cauchy = solution_cauchy[circle_var]
weights_cauchy = compute_scalar_weights(
    params_cauchy, points, jaxls.utils.irls_cauchy_adaptive()
)

print(
    f"Adaptive Cauchy:  center=({params_cauchy[0]:.3f}, {params_cauchy[1]:.3f}), "
    f"radius={params_cauchy[2]:.3f}"
)
print(
    f"True parameters: center=({true_cx:.3f}, {true_cy:.3f}), radius={true_r:.3f}"
)
print(
    f"\nCenter error: "
    f"{jnp.sqrt((params_cauchy[0] - true_cx) ** 2 + (params_cauchy[1] - true_cy) ** 2):.4f}"
)
print(f"Radius error:  {jnp.abs(params_cauchy[2] - true_r):.4f}")

INFO     | Building optimization problem with 50 terms and 1 variables: 50 costs, 0 eq_zero, 0 leq_zero, 0 geq_zero
INFO     | Vectorizing group with 50 costs, 1 variables each: robust_circle_residual
INFO     | Variable elimination: eliminating CircleVar (3 of 3 tangent dims); reduced system is 0-dimensional
INFO     | Variable elimination: every type was eliminated, so the Hessian is fully block-diagonal; the step is an exact blockwise inverse and the linear_solver choice is ignored.
INFO     | The Hessian is fully block-diagonal (every variable type was eliminated), so linear_solver='conjugate_gradient' is ignored; the step is solved by exact blockwise inversion.
INFO     |  step #0: cost=19.7902 lambd=0.0005 inexact_tol=1.0e-02
INFO     |      - robust_circle_residual(50): 19.79016 (avg 0.19790)
INFO     |      accepted=True ATb_norm=1.32e+01 cost_prev=19.7902 cost_new=3.4020
INFO     |  step #1: cost=3.4020 lambd=0.0003 inexact_tol=1.0e-02
INFO     |      - robust_circle_residual(

## Visualization

Compare standard least squares (pulled by outliers) vs adaptive IRLS (robust
to outliers):

In [10]:
from plotly.subplots import make_subplots

fig_compare = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Standard Least Squares", "Adaptive IRLS (Cauchy)"),
)

for col in [1, 2]:
    fig_compare.add_trace(
        go.Scatter(
            x=all_x[~is_outlier],
            y=all_y[~is_outlier],
            mode="markers",
            marker=dict(size=8, color="#2196F3"),
            name="Inliers",
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
    )
    fig_compare.add_trace(
        go.Scatter(
            x=all_x[is_outlier],
            y=all_y[is_outlier],
            mode="markers",
            marker=dict(size=10, color="#F44336", symbol="x"),
            name="Outliers",
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
    )
    true_circle = make_circle_trace(
        true_cx, true_cy, true_r, "True circle", "#4CAF50", "dash"
    )
    true_circle.showlegend = col == 1
    fig_compare.add_trace(true_circle, row=1, col=col)

# Standard LS result.
fig_compare.add_trace(
    make_circle_trace(
        float(params_standard[0]),
        float(params_standard[1]),
        float(params_standard[2]),
        "Standard LS",
        "#FF9800",
    ),
    row=1,
    col=1,
)

# Adaptive IRLS result.
fig_compare.add_trace(
    make_circle_trace(
        float(params_cauchy[0]),
        float(params_cauchy[1]),
        float(params_cauchy[2]),
        "Adaptive Cauchy",
        "#9C27B0",
    ),
    row=1,
    col=2,
)

fig_compare.update_xaxes(title_text="x", scaleanchor="y", scaleratio=1)
fig_compare.update_yaxes(title_text="y")
fig_compare.update_layout(
    height=450,
    margin=dict(t=40, b=40, l=60, r=40),
    legend=dict(
        orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5
    ),
)
HTML(fig_compare.to_html(full_html=False, include_plotlyjs="cdn"))

## Weight visualization

IRLS identifies outliers by assigning them low weights.  Points are coloured
by their adaptive Cauchy weight (blue = high weight / inlier, red = low
weight / outlier).  Because the scale σ is inferred from the residuals, even
if the absolute residuals change between problems, the relative weighting
remains well-calibrated.

In [11]:
fig_w = go.Figure()

fig_w.add_trace(
    go.Scatter(
        x=all_x,
        y=all_y,
        mode="markers",
        marker=dict(
            size=12,
            color=weights_cauchy,
            colorscale=[[0, "#F44336"], [1, "#2196F3"]],
            colorbar=dict(title="Weight", thickness=15),
            cmin=0,
            cmax=1,
        ),
        text=[f"Weight: {w:.3f}" for w in weights_cauchy],
        hovertemplate="(%{x:.2f}, %{y:.2f})<br>%{text}<extra></extra>",
        name="Points",
    )
)
fig_w.add_trace(
    make_circle_trace(
        float(params_cauchy[0]),
        float(params_cauchy[1]),
        float(params_cauchy[2]),
        "Adaptive Cauchy fit",
        "#9C27B0",
    )
)
fig_w.add_trace(
    make_circle_trace(true_cx, true_cy, true_r, "True circle", "#4CAF50", "dash")
)

fig_w.update_xaxes(title_text="x", scaleanchor="y", scaleratio=1)
fig_w.update_yaxes(title_text="y")
fig_w.update_layout(
    title="Points Coloured by Adaptive Cauchy IRLS Weight",
    height=450,
    margin=dict(t=60, b=40, l=60, r=40),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
)
HTML(fig_w.to_html(full_html=False, include_plotlyjs="cdn"))

## Comparing all six adaptive M-estimators

Run all six adaptive factories and compare to the standard-LS baseline.
All use their default parameters and require **zero manual tuning**:

In [12]:
results = {"Standard LS": params_standard}

for name, weight_fn in [
    ("Huber", jaxls.utils.irls_huber_adaptive()),
    ("Cauchy", jaxls.utils.irls_cauchy_adaptive()),
    ("Tukey", jaxls.utils.irls_tukey_adaptive()),
    ("Geman-McClure", jaxls.utils.irls_geman_mcclure_adaptive()),
    ("Welsh", jaxls.utils.irls_welsh_adaptive()),
    ("Lp (p=1.2)", jaxls.utils.irls_lp_adaptive(p=1.2)),
]:
    cost = make_adaptive_robust_circle_cost(weight_fn)
    costs = [
        cost(CircleVar(id=jnp.zeros(n_points, dtype=jnp.int32)), points)
    ]
    problem = jaxls.LeastSquaresProblem(costs, [circle_var]).analyze()
    solution = problem.solve(initial_vals, verbose=False)
    results[name] = solution[circle_var]

print(
    f"{'Method':<22} {'cx':>8} {'cy':>8} {'r':>8}"
    f" {'Center err':>12} {'Radius err':>12}"
)
print("-" * 76)
print(
    f"{'True':<22} {true_cx:>8.3f} {true_cy:>8.3f} {true_r:>8.3f}"
    f" {'-':>12} {'-':>12}"
)
for name, params in results.items():
    cerr = float(
        jnp.sqrt((params[0] - true_cx) ** 2 + (params[1] - true_cy) ** 2)
    )
    rerr = float(jnp.abs(params[2] - true_r))
    print(
        f"{name:<22} {float(params[0]):>8.3f} {float(params[1]):>8.3f}"
        f" {float(params[2]):>8.3f} {cerr:>12.4f} {rerr:>12.4f}"
    )

INFO     | Building optimization problem with 50 terms and 1 variables: 50 costs, 0 eq_zero, 0 leq_zero, 0 geq_zero
INFO     | Vectorizing group with 50 costs, 1 variables each: robust_circle_residual
INFO     | Variable elimination: eliminating CircleVar (3 of 3 tangent dims); reduced system is 0-dimensional
INFO     | Variable elimination: every type was eliminated, so the Hessian is fully block-diagonal; the step is an exact blockwise inverse and the linear_solver choice is ignored.
INFO     | Building optimization problem with 50 terms and 1 variables: 50 costs, 0 eq_zero, 0 leq_zero, 0 geq_zero
INFO     | Vectorizing group with 50 costs, 1 variables each: robust_circle_residual
INFO     | Variable elimination: eliminating CircleVar (3 of 3 tangent dims); reduced system is 0-dimensional
INFO     | Variable elimination: every type was eliminated, so the Hessian is fully block-diagonal; the step is an exact blockwise inverse and the linear_solver choice is ignored.
INFO     | Buildin

In [13]:
colors = {
    "Standard LS": "#FF9800",
    "Huber": "#2196F3",
    "Cauchy": "#4CAF50",
    "Tukey": "#00BCD4",
    "Geman-McClure": "#E91E63",
    "Welsh": "#9C27B0",
    "Lp (p=1.2)": "#795548",
}

fig_all = go.Figure()

fig_all.add_trace(
    go.Scatter(
        x=all_x[~is_outlier],
        y=all_y[~is_outlier],
        mode="markers",
        marker=dict(size=8, color="#607D8B"),
        name="Inliers",
    )
)
fig_all.add_trace(
    go.Scatter(
        x=all_x[is_outlier],
        y=all_y[is_outlier],
        mode="markers",
        marker=dict(size=10, color="#F44336", symbol="x"),
        name="Outliers",
    )
)
fig_all.add_trace(
    make_circle_trace(true_cx, true_cy, true_r, "True", "#4CAF50", "dash")
)

for name, params in results.items():
    fig_all.add_trace(
        make_circle_trace(
            float(params[0]),
            float(params[1]),
            float(params[2]),
            name,
            colors[name],
        )
    )

fig_all.update_xaxes(title_text="x", scaleanchor="y", scaleratio=1)
fig_all.update_yaxes(title_text="y")
fig_all.update_layout(
    title="Comparison of All Adaptive M-Estimators for Circle Fitting",
    height=500,
    margin=dict(t=60, b=40, l=60, r=40),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
)
HTML(fig_all.to_html(full_html=False, include_plotlyjs="cdn"))

## Adaptive vs fixed-scale: robustness under varying conditions

A fixed-scale parameter (e.g. `irls_huber(delta=0.5)`) works well **only
when the threshold matches the residual scale**.  When conditions change —
different units, a rescaled problem, or heavier outlier contamination — the
same threshold can be too tight (treating inliers as outliers) or too loose
(letting outliers through).  The adaptive estimator re-estimates σ at every
iteration, so it handles all these cases without retuning.

We compare four methods across three scenarios using the **Huber** kernel,
whose hard threshold makes scale mismatch especially visible:

| Scenario | Description | Challenge for fixed `delta=0.5` |
|----------|-------------|-----------------------------|
| **A. Baseline** | σ=0.1, 20 % outliers | `delta=0.5` is well-tuned — both methods work |
| **B. 10× scale** | Same geometry ×10 (residuals ~10× larger) | Even *inlier* residuals exceed `delta=0.5` → all points get down-weighted |
| **C. Heavy outliers** | σ=0.1, 40 % outliers | Extreme contamination; robust methods stressed |

In [14]:
def generate_circle_data_ex(n_inliers, n_outliers, noise_std, scale=1.0, seed=42):
    """Generate circle data with configurable noise, outlier count, and scale."""
    rng = np.random.RandomState(seed)
    theta = rng.uniform(0, 2 * np.pi, n_inliers)
    noise = rng.normal(0, noise_std, n_inliers)
    gt_cx, gt_cy, gt_r = true_cx * scale, true_cy * scale, true_r * scale
    ix = gt_cx + (gt_r + noise * scale) * np.cos(theta)
    iy = gt_cy + (gt_r + noise * scale) * np.sin(theta)
    lo, hi = -4 * scale, 7 * scale
    ox = rng.uniform(lo, hi, n_outliers)
    oy = rng.uniform(lo, hi, n_outliers)
    px = np.concatenate([ix, ox])
    py = np.concatenate([iy, oy])
    pts = jnp.stack([px, py], axis=-1)
    outlier_mask = np.array([False] * n_inliers + [True] * n_outliers)
    return pts, px, py, outlier_mask, (gt_cx, gt_cy, gt_r)


def solve_scenario_ex(pts, gt, irls_weight_fn=None):
    """Solve circle fitting and return (params, center_err, radius_err)."""
    gt_cx, gt_cy, gt_r = gt
    n = len(pts)
    cent = jnp.mean(pts, axis=0)
    avg_d = jnp.mean(jnp.sqrt(jnp.sum((pts - cent) ** 2, axis=-1)))
    init = jaxls.VarValues.make(
        [circle_var.with_value(jnp.array([cent[0], cent[1], avg_d]))]
    )
    if irls_weight_fn is not None:
        cost_fn = make_adaptive_robust_circle_cost(irls_weight_fn)
    else:
        cost_fn = circle_residual
    costs = [cost_fn(CircleVar(id=jnp.zeros(n, dtype=jnp.int32)), pts)]
    prob = jaxls.LeastSquaresProblem(costs, [circle_var]).analyze()
    sol = prob.solve(init, verbose=False)
    p = sol[circle_var]
    cerr = float(jnp.sqrt((p[0] - gt_cx) ** 2 + (p[1] - gt_cy) ** 2))
    rerr = float(jnp.abs(p[2] - gt_r))
    return p, cerr, rerr


# Define scenarios.
scen_a = generate_circle_data_ex(40, 10, 0.1, scale=1.0)
scen_b = generate_circle_data_ex(40, 10, 0.1, scale=10.0)
scen_c = generate_circle_data_ex(30, 20, 0.1, scale=1.0, seed=13)

scenarios = {
    "A. Baseline\n(σ=0.1, 20% outliers)": scen_a,
    "B. 10× scale\n(same geometry ×10)": scen_b,
    "C. 40% outliers\n(σ=0.1)": scen_c,
}

# Methods: fixed delta=0.5 tuned for Scenario A, plus adaptive.
methods = {
    "Standard LS":       None,
    "Fixed (δ=0.5)":     jaxls.utils.irls_huber(delta=0.5),
    "Fixed (δ=5.0)":     jaxls.utils.irls_huber(delta=5.0),
    "Adaptive Huber":    jaxls.utils.irls_huber_adaptive(),
}

# Run all combinations.
all_results = {}
for sname, (pts_s, _, _, _, gt_s) in scenarios.items():
    for mname, wfn in methods.items():
        all_results[(sname, mname)] = solve_scenario_ex(pts_s, gt_s, wfn)

# Print results table.
print(f"{'Scenario':<35} {'Method':<20} {'Center err':>11} {'Radius err':>11}")
print("=" * 80)
for sname in scenarios:
    label = sname.replace('\n', ' ')
    for i, mname in enumerate(methods):
        _, cerr, rerr = all_results[(sname, mname)]
        row_label = label if i == 0 else ""
        print(f"{row_label:<35} {mname:<20} {cerr:>11.4f} {rerr:>11.4f}")
    print()

INFO     | Building optimization problem with 50 terms and 1 variables: 50 costs, 0 eq_zero, 0 leq_zero, 0 geq_zero
INFO     | Vectorizing group with 50 costs, 1 variables each: circle_residual
INFO     | Variable elimination: eliminating CircleVar (3 of 3 tangent dims); reduced system is 0-dimensional
INFO     | Variable elimination: every type was eliminated, so the Hessian is fully block-diagonal; the step is an exact blockwise inverse and the linear_solver choice is ignored.
INFO     | Building optimization problem with 50 terms and 1 variables: 50 costs, 0 eq_zero, 0 leq_zero, 0 geq_zero
INFO     | Vectorizing group with 50 costs, 1 variables each: robust_circle_residual
INFO     | Variable elimination: eliminating CircleVar (3 of 3 tangent dims); reduced system is 0-dimensional
INFO     | Variable elimination: every type was eliminated, so the Hessian is fully block-diagonal; the step is an exact blockwise inverse and the linear_solver choice is ignored.
INFO     | Building optim

In [15]:
method_colors = {
    "Standard LS":      "#FF9800",
    "Fixed (δ=0.5)":    "#2196F3",
    "Fixed (δ=5.0)":    "#00BCD4",
    "Adaptive Huber":   "#9C27B0",
}

fig_cmp = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        s.replace('\\n', '<br>').replace('\n', '<br>') for s in scenarios
    ],
)

for col_idx, (sname, (pts_s, px_s, py_s, mask_s, gt_s)) in enumerate(
    scenarios.items(), 1
):
    gt_cx_s, gt_cy_s, gt_r_s = gt_s
    show_legend = col_idx == 1

    fig_cmp.add_trace(
        go.Scatter(
            x=px_s[~mask_s], y=py_s[~mask_s],
            mode="markers", marker=dict(size=6, color="#607D8B"),
            name="Inliers", showlegend=show_legend,
        ), row=1, col=col_idx,
    )
    fig_cmp.add_trace(
        go.Scatter(
            x=px_s[mask_s], y=py_s[mask_s],
            mode="markers", marker=dict(size=8, color="#F44336", symbol="x"),
            name="Outliers", showlegend=show_legend,
        ), row=1, col=col_idx,
    )

    tc = make_circle_trace(
        gt_cx_s, gt_cy_s, gt_r_s, "True", "#4CAF50", "dash"
    )
    tc.showlegend = show_legend
    fig_cmp.add_trace(tc, row=1, col=col_idx)

    for mname, color in method_colors.items():
        p, _, _ = all_results[(sname, mname)]
        tr = make_circle_trace(
            float(p[0]), float(p[1]), float(p[2]), mname, color,
        )
        tr.showlegend = show_legend
        fig_cmp.add_trace(tr, row=1, col=col_idx)

fig_cmp.update_xaxes(title_text="x", scaleanchor="y", scaleratio=1)
fig_cmp.update_yaxes(title_text="y")
fig_cmp.update_layout(
    title="Fixed-Scale vs Adaptive Huber IRLS Under Different Conditions",
    height=450,
    margin=dict(t=80, b=40, l=60, r=40),
    legend=dict(
        orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5,
    ),
)
HTML(fig_cmp.to_html(full_html=False, include_plotlyjs="cdn"))

**Key observations:**

- **Scenario A** (baseline): `Fixed (δ=0.5)` is well-tuned for the
  inlier noise σ=0.1, so it works well.  `Fixed (δ=5.0)` is too loose —
  outlier residuals of ~2–5 slip under the threshold.  Adaptive Huber
  estimates σ from the data and sets the effective threshold
  $k \cdot \hat{\sigma} \approx 1.345 \times 0.15 \approx 0.2$, close to
  the optimal.

- **Scenario B** (10× scale): This is the *same* geometric problem, just
  expressed in different units.  `Fixed (δ=0.5)` now treats every point
  as an outlier (inlier residuals ~1.0 are all above 0.5), destroying the
  fit.  `Fixed (δ=5.0)` happens to work for this scale.  Adaptive Huber
  re-estimates σ ≈ 1.5 and works just as well as in Scenario A — **it is
  scale-invariant by design**.

- **Scenario C** (40 % outliers): Heavy contamination stresses all
  methods.  Adaptive Huber still works because MAD remains a valid scale
  estimate up to ~50 % contamination.

**Takeaway:** No single fixed threshold works across all scenarios.
`δ=0.5` is great for Scenario A but catastrophic for B; `δ=5.0` handles
B but is too loose for A.  **Adaptive IRLS delivers consistent
performance across all scenarios with zero tuning** — it adapts the
effective threshold to whatever residual scale the data presents.

## Summary

### How to use adaptive IRLS in jaxls

1. **Choose an adaptive factory** from `jaxls.utils`:

   | Factory | Behaviour | When to use |
   |---------|-----------|-------------|
   | `irls_huber_adaptive(k=1.345)` | Quadratic near centre, linear tails | General-purpose; convex, so convergence is stable |
   | `irls_cauchy_adaptive(k=2.385)` | Smooth down-weighting | Moderate outlier contamination; never fully rejects |
   | `irls_tukey_adaptive(k=4.685)` | Hard rejection beyond $k\hat{\sigma}$ | Heavy contamination; strongest rejection but needs good initialisation |
   | `irls_geman_mcclure_adaptive(k=3.0)` | Squared Cauchy decay | Very aggressive rejection; redescending influence |
   | `irls_welsh_adaptive(k=2.985)` | Exponential decay | Fast fall-off like Tukey, but smooth (never exactly zero) |
   | `irls_lp_adaptive(p=1.2)` | Power-law $|u|^{p-2}$ | Tunable robustness via $p$; $p{=}1$ is L1, $p{=}2$ is L2 |

2. **Pass it to `Cost.factory`**:

   ```python
   @jaxls.Cost.factory(irls_weight_fn=jaxls.utils.irls_cauchy_adaptive())
   def my_cost(vals, var, data):
       return plain_residual(vals, var, data)  # no weighting here
   ```

3. **Solve as usual** — the solver handles IRLS internally:
   - At each iteration, `irls_weight_fn` receives the full `(count, residual_flat_dim)` residual array.
   - The adaptive factories estimate $\hat{\sigma}$ via MAD, normalise, and return weights.
   - The solver multiplies both $r$ and $J$ by $\sqrt{w}$, so the normal equations become $J^T W J\,\Delta x = -J^T W r$.

### Adaptive vs fixed-parameter

| | Fixed (e.g. `irls_cauchy(c=0.5)`) | Adaptive (e.g. `irls_cauchy_adaptive()`) |
|---|---|---|
| Scale parameter | You choose, in residual units | Estimated from data each iteration |
| Tuning effort | Needs domain knowledge | None — use default $k$ |
| Different noise levels | Fixed threshold | Automatically re-scales |
| Implementation | Manual `stop_gradient` in residual | Built-in `irls_weight_fn` API |

### Fixed-scale weight functions

For cases where you know the noise scale a priori, jaxls also provides
fixed-parameter factories that can be used identically via `irls_weight_fn`:

- `jaxls.utils.irls_huber(delta)` — Huber with fixed threshold $\delta$
- `jaxls.utils.irls_cauchy(c)` — Cauchy with fixed scale $c$
- `jaxls.utils.irls_tukey(c)` — Tukey bisquare with fixed threshold $c$
- `jaxls.utils.irls_geman_mcclure(c)` — Geman-McClure with fixed scale $c$
- `jaxls.utils.irls_welsh(c)` — Welsh with fixed scale $c$
- `jaxls.utils.irls_lp(p, eps)` — L_p norm with fixed exponent $p$
- `jaxls.utils.irls_l1(eps)` — L1 approximation ($w = 1/|r|$)

See the [fixed-parameter robust costs notebook](robust_costs.ipynb) for
the manual `stop_gradient` approach.

For more details, see {class}`jaxls.Var`, {class}`jaxls.Cost`, and
{class}`jaxls.LeastSquaresProblem`.